# CRISP-DM Fase 6: Deployment & Communication - Visualizaciones Ejecutivas
## Dataset: Documentación Técnica y Boletines Webservices SIPSA (DANE)

**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 6: Deployment & Business Insights) | **Fase PDCO**: OPERATIONS  
**Estándares**: Data Storytelling, Diagramas de Control Nelson, Toma de Decisiones Agroempresariales.

---

### 📊 1. Tableros Ejecutivos de Visualización
* Comunicación gráfica rigurosa para directores gremiales, agricultores y formuladores de política pública.
* Gráficos con bandas de confianza del 95% y límites de control superior e inferior ($\\mu \\pm 3\\sigma$).
* Cuadros de mando con KPIs de síntesis para responder a las preguntas de la Batería A1-J1.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [FASE 8: GENERACIÓN DE GRÁFICOS EJECUTIVOS PARA TOMA DE DECISIONES]
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
from src.database.db_manager import DatabaseManager

db = DatabaseManager(str(APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"))
df = db.query("SELECT * FROM doc_webservice_chunks")

primary_col = "char_length"
group_col = "source_file"

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if primary_col in df.columns and pd.api.types.is_numeric_dtype(df[primary_col]):
    # 1. Gráfico de Control y Tendencia (Series)
    s = df[primary_col].dropna().reset_index(drop=True)
    mu, sigma = s.mean(), s.std()
    
    axes[0].plot(s.index[:100], s.values[:100], color="#1d3557", lw=1.5, label="Observaciones")
    axes[0].axhline(mu, color="#2a9d8f", linestyle="-", label=f"Media: {mu:,.2f}")
    axes[0].axhline(mu + 3*sigma, color="#e63946", linestyle="--", label="Límite Superior (+3σ)")
    axes[0].axhline(max(0, mu - 3*sigma), color="#e63946", linestyle="--", label="Límite Inferior (-3σ)")
    axes[0].fill_between(s.index[:100], max(0, mu - 2*sigma), mu + 2*sigma, color="#a8dadc", alpha=0.3, label="Zona Control 95%")
    axes[0].set_title(f"Carta de Control Estadístico: {primary_col} (Documentación Técnica y Boletines Webservices SIPSA (DANE))", fontsize=11, fontweight="bold")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_ylabel(primary_col)
    
    # 2. Gráfico por Categorías / Nodos Territoriales
    if group_col in df.columns:
        top_cats = df.groupby(group_col)[primary_col].median().sort_values(ascending=False).head(8)
        sns.barplot(x=top_cats.values, y=top_cats.index, ax=axes[1], palette="crest")
        axes[1].set_title(f"Mediana por {group_col} (Top 8 Nodos)", fontsize=11, fontweight="bold")
        axes[1].set_xlabel("Mediana " + primary_col)
else:
    axes[0].text(0.5, 0.5, "Dataset Estructurado Documental", ha="center", va="center")
    axes[1].text(0.5, 0.5, "Revisión de Frecuencias Textuales", ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# [FASE 8: CUADRO DE MANDO Y RECOMENDACIONES DE NEGOCIO (CRISP-DM)]
# ==============================================================================
print("=" * 70)
print(f"🌾 SÍNTESIS EJECUTIVA DE NEGOCIO: Documentación Técnica y Boletines Webservices SIPSA (DANE)")
print("=" * 70)
print(f"• Objetivo de Negocio: Automatizar la extracción de especificaciones técnicas y esquemas de datos a partir de manuales y boletines institucionales.")
print("\n💡 RECOMENDACIONES DE OPERACIÓN:")
print("1. Implementar cobertura financiera o contratos forward ante alertas de volatilidad.")
print("2. Priorizar la articulación logística entre municipios emisores y centrales receptoras.")
print("3. Monitorear continuamente los umbrales de alerta temprana en la plataforma AgroStatsApp.")
print("=" * 70)
print("✅ CICLO DE VIDA CRISP-DM COMPLETADO CON ÉXITO.")
